In [43]:
import os
import polars as pl
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

### Wczytywanie danych

In [34]:
file_path = str(Path.home() / "Desktop" / "Hackaton VISA" / "datasprint_sample_data.parquet")

### Czyszczenie danych

In [18]:
lau = duckdb.sql(f"""WITH postal_counts AS (
    SELECT pymt_crd_acct_num_raw
        , pstl_cd_enr
        , lau_enr
        , fua_enr
        , COUNT(*) AS cnt
    FROM '{file_path}'
    WHERE pstl_cd_enr IS NOT NULL
    GROUP BY
        pymt_crd_acct_num_raw
        , pstl_cd_enr
        , lau_enr
        , fua_enr
)

, ranked AS (
    SELECT
        pymt_crd_acct_num_raw
        , pstl_cd_enr
        , lau_enr
        , fua_enr
        , cnt
        , ROW_NUMBER() OVER (
            PARTITION BY pymt_crd_acct_num_raw
            ORDER BY cnt DESC
        ) AS rn
    FROM postal_counts
)

SELECT
    pymt_crd_acct_num_raw
    , pstl_cd_enr
    , lau_enr
    , fua_enr
    , cnt AS postal_code_tx
FROM ranked 
WHERE rn = 1;""").to_df()

In [19]:
lau

,pymt_crd_acct_num_raw,pstl_cd_enr,lau_enr,fua_enr,postal_code_tx
0,1238842041d48fee3519827cb3a34c7f3f79f2ba4cd9cee,55-320,MALCZYCE,LEGNICA,72
1,4880081f9df3211c07d9ee5008bb8b0cc99674e0dda1cbb,02-903,WARSZAWA,WARSZAWA,54
2,70f291f00740a155b65acca9ccf083f622e47e960af98b1,44-190,KNUROW,KATOWICE,115
3,31a31934db0fb0e3804158df151622fa4d97b4fe7f15432,46-060,OPOLE,OPOLE,46
4,891b61db1d4cc64abf28ff20d1f694ac22b7c487747f5a7,02-903,WARSZAWA,WARSZAWA,186
...,...,...,...,...,...
1276486,b0e9852d4b143e331f1fcdd4d7ebb53e59ae02d5588e888,98-300,WIELUN,CZESTOCHOWA,4
1276487,a082e4b4df42a7c143701924627bd41cdb1a9f30c30efa0,40-017,KATOWICE,KATOWICE,16
1276488,9f62a17c2b9f1400868fc46818f692087559e2ec766ba25,43-108,TYCHY,KATOWICE,28
1276489,d191fb697edfe44ea677023cef5f06d042847d03fbf172e,02100,GODOLLO,BUDAPEST,9


In [8]:
country = 616
city = 'KRAKOW%'

In [27]:
lau_2 = duckdb.sql(f" SELECT pymt_crd_acct_num_raw, pstl_cd_enr, lau_enr FROM lau WHERE fua_enr LIKE '{city}'").to_df()

In [28]:
lau_2

,pymt_crd_acct_num_raw,pstl_cd_enr,lau_enr
0,5792f32aeeb94c64157a4e7834d7b2af7a1d333a4ac180b,30-701,KRAKOW
1,127f54bde2af009985189032f0cf4eb32bf3fc4fc8c1193,31-510,KRAKOW
2,c2840ae5556974b3823e2aea888e762413e4408b0d5ec4d,32-104,KONIUSZA
3,3f95f176397c1f51f0d363838f5d0892880c505f3b624b1,31-625,KRAKOW
4,4663ce11bc5b3faab94248b61ce57cb66e21d23348567f0,31-564,KRAKOW
...,...,...,...
66513,028af6128e284c0d56ff256dd408f81728756704daac31a,32-100,PROSZOWICE
66514,46f53eb505a2340e1c2f8036b62484db5d84bbd2993e970,32-065,KRZESZOWICE
66515,4a126f13b816aa682572920add2f21f7186317f3d9cda54,32-060,LISZKI
66516,438ed9c30707f0d7d686e8c03d7b6bdbb323bf70faf8722,32-010,KOCMYRZOW-LUBORZYCA


In [94]:
lau_2["pstl_cd_enr"] = lau_2["pstl_cd_enr"].apply(
    lambda x: x if not isinstance(x, str)
    else (x if len(x) == 6 and x[2] == "-" else x[:2] + "-" + x[-3:])
)

In [95]:
sprzedawcy = duckdb.sql(f" SELECT tran_id_raw, pymt_crd_acct_num_raw, transaction_type, upper(mrch_nm_raw) as mrch_nm_raw, mrch_catg_cd, mrch_catg_nm, mrch_postal_code, cs_tran_amt, prch_mnth_id FROM '{file_path}' WHERE prod_id_pltfrm_cd_vcis = 'CN' and cp_flag = 1 and mrch_ctry_cd = 616 and mrch_city_nm_raw LIKE '{city}'").to_df() 

In [96]:
sprzedawcy.head()

,tran_id_raw,pymt_crd_acct_num_raw,transaction_type,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_postal_code,cs_tran_amt,prch_mnth_id
0,974298192,72f97cee703ab09d7127e5995a422d107a8707fbb4320cb,POS,ZABKA ZB263 K.1,5499,MISC FOOD STORES - DEFAULT,31-476,139.04880,202506
1,974751128,4cf266a05a8f23b67bb8cf1f4ec4780120ca06616ddc020,POS,PIEKARNIA U BRACI,5462,BAKERIES,31-625,160.74240,202507
2,974751357,d318243e56ac2b8a967413547b8dc321d1c5bed3bf9235c,POS,OYSHO BONARKA 16044,5691,MENS/WOMENS CLOTHING STORES,31-625,363.30800,202507
3,974751421,8bcdf72ae320e4e53140ad06840abfd709b982a99dbea25,POS,DM-DROGERIE MARKT 022,5912,DRUG STORES & PHARMACIES,31-801,92.39560,202507
4,974751580,c0896b8b90728a38562c964d3e1b4932b4ab96d11689fec,NA,LPP RESERVED 1912217,5651,FAMILY CLOTHING STORES,31-123,154.54344,202507


In [98]:
csv_path = str(Path.home() / "Desktop" / "Hackaton VISA" / "kody_dzielnice.csv")

In [99]:
dzielnice = pd.read_csv(csv_path)

In [100]:
dzielnice

,pstl_cd,district
0,21-586,Dzielnica XIV Czyżyny
1,30-000,Dzielnica IV Prądnik Biały
2,30-015,Dzielnica V Krowodrza
3,30-058,Dzielnica V Krowodrza
4,30-059,Dzielnica V Krowodrza
...,...,...
1609,37-401,Dzielnica XII Bieżanów-Prokocim
1610,39-398,Dzielnica VIII Dębniki
1611,39-648,Dzielnica XI Podgórze Duchackie
1612,39-686,Dzielnica XI Podgórze Duchackie


In [102]:
dzielnice_kup = duckdb.sql(f" SELECT pymt_crd_acct_num_raw, pstl_cd_enr, lau_enr, district FROM lau_2 LEFT JOIN dzielnice ON pstl_cd_enr = pstl_cd WHERE district IS NOT NULL").to_df()

In [103]:
sprzedawcy["mrch_postal_code"] = sprzedawcy["mrch_postal_code"].apply(
    lambda x: x if not isinstance(x, str)
    else (x if len(x) == 6 and x[2] == "-" else x[:2] + "-" + x[-3:])
)

In [104]:
sprzedawcy

,tran_id_raw,pymt_crd_acct_num_raw,transaction_type,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_postal_code,cs_tran_amt,prch_mnth_id
0,974298192,72f97cee703ab09d7127e5995a422d107a8707fbb4320cb,POS,ZABKA ZB263 K.1,5499,MISC FOOD STORES - DEFAULT,31-476,139.04880,202506
1,974751128,4cf266a05a8f23b67bb8cf1f4ec4780120ca06616ddc020,POS,PIEKARNIA U BRACI,5462,BAKERIES,31-625,160.74240,202507
2,974751357,d318243e56ac2b8a967413547b8dc321d1c5bed3bf9235c,POS,OYSHO BONARKA 16044,5691,MENS/WOMENS CLOTHING STORES,31-625,363.30800,202507
3,974751421,8bcdf72ae320e4e53140ad06840abfd709b982a99dbea25,POS,DM-DROGERIE MARKT 022,5912,DRUG STORES & PHARMACIES,31-801,92.39560,202507
4,974751580,c0896b8b90728a38562c964d3e1b4932b4ab96d11689fec,NA,LPP RESERVED 1912217,5651,FAMILY CLOTHING STORES,31-123,154.54344,202507
...,...,...,...,...,...,...,...,...,...
6597376,90336874,01bf3e55d2ba293a34f5175fc8a1eaecb8485b975744b1e,POS,PAUZA - HUB,5814,FAST FOOD RESTAURANTS,31-625,106.35200,202507
6597377,90337008,0a9f4b751a489c091faae070e2c9aa373334193482338cd,POS,ZABKA Z8575 K.1,5499,MISC FOOD STORES - DEFAULT,31-315,16.80840,202507
6597378,90454922,531f67c01c1461e6b1adf0c7d0f2d940286dd09bd3f2c1e,POS,RAMEN PEOPLE,5812,EATING PLACES AND RESTAURANTS,31-625,119.78400,202602
6597379,90454988,fee5bf87b6274101c953c0ea58fec265dc3a29e4ee510f1,POS,PRACE PLASTYCZNE,5947,"GIFT, CARD, NOVELTY STORES",31-042,52.44000,202602


In [105]:
dzielnice_kup

,pymt_crd_acct_num_raw,pstl_cd_enr,lau_enr,district
0,5792f32aeeb94c64157a4e7834d7b2af7a1d333a4ac180b,30-701,KRAKOW,Dzielnica VIII Dębniki
1,127f54bde2af009985189032f0cf4eb32bf3fc4fc8c1193,31-510,KRAKOW,Dzielnica I Stare Miasto
2,3f95f176397c1f51f0d363838f5d0892880c505f3b624b1,31-625,KRAKOW,Dzielnica XV Mistrzejowice
3,4663ce11bc5b3faab94248b61ce57cb66e21d23348567f0,31-564,KRAKOW,Dzielnica II Grzegórzki
4,a4927cd6925ef8f97467843216153af71624b3f8d2a4f38,31-586,KRAKOW,Dzielnica XIV Czyżyny
...,...,...,...,...
46379,7952455692dade84ddb994c2ab141c0739d8d817cd832ac,32-050,SKAWINA,Dzielnica IX Łagiewniki-Borek Fałęcki
46380,636adfcd898e270084e325d0831a3da73a73f5c676fe764,31-510,KRAKOW,Dzielnica I Stare Miasto
46381,16f00372e7975d23cf16c81872f4f33a2d9a1580ae02bff,32-060,LISZKI,Dzielnica I Stare Miasto
46382,4a126f13b816aa682572920add2f21f7186317f3d9cda54,32-060,LISZKI,Dzielnica I Stare Miasto


In [106]:
dzielnice_sprzed = duckdb.sql(f""" SELECT 
    tran_id_raw, 
    pymt_crd_acct_num_raw, 
    transaction_type, 
    upper(mrch_nm_raw) as mrch_nm_raw, 
    mrch_catg_cd, mrch_catg_nm, 
    mrch_postal_code, 
    district,
    cs_tran_amt, 
    prch_mnth_id
    
    FROM sprzedawcy 
    
    LEFT JOIN dzielnice ON mrch_postal_code = pstl_cd 
    WHERE district IS NOT NULL""").to_df() 

In [107]:
dzielnice_sprzed

,tran_id_raw,pymt_crd_acct_num_raw,transaction_type,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_postal_code,district,cs_tran_amt,prch_mnth_id
0,564237693,8e921b0e1af6b93c8affdb7e9bf212881453b4c10f74ea7,POS,HALA CENTRALNA,5812,EATING PLACES AND RESTAURANTS,31-625,Dzielnica XV Mistrzejowice,24.840000,202511
1,564237694,8e921b0e1af6b93c8affdb7e9bf212881453b4c10f74ea7,POS,HALA CENTRALNA,5812,EATING PLACES AND RESTAURANTS,31-625,Dzielnica XV Mistrzejowice,10.488000,202511
2,452877861,406ed7b3d2c7e10611d8434c285ff1e111e25708ead4c8c,POS,KAUFLAND 7062,5411,GROCERY STORES/SUPERMARKETS,31-625,Dzielnica XV Mistrzejowice,484.161040,202602
3,452878353,70d978108df74b25bb1dcf2f958de8beb14967a0c2965d0,POS,CARREFOUR EXPRESS,5411,GROCERY STORES/SUPERMARKETS,31-133,Dzielnica I Stare Miasto,6.397680,202602
4,452335210,31179ea9e7b09100a1cfaeaef320d4fedf6de1310834ffc,POS,JMP S.A. BIEDRONKA 3647,5411,GROCERY STORES/SUPERMARKETS,30-758,Dzielnica XII Bieżanów-Prokocim,139.852880,202512
...,...,...,...,...,...,...,...,...,...,...
6371462,1444449113,42dd7f3767e0850dd240d8bd64ea29d42f4c71954025c5b,POS,KASOWNIK ZTP KRAKOW K1610,4111,LOCAL COMMUTER TRANSPORT,31-625,Dzielnica XV Mistrzejowice,18.768000,202605
6371463,66704046,90a5fb2603e05772bb066a7316fb148d9a5667d2e4fc8aa,POS,ZABKA Z5482 K.2,5499,MISC FOOD STORES - DEFAULT,31-534,Dzielnica II Grzegórzki,10.099760,202501
6371464,66704079,3923cb33fd4b09babcb06e4004629df9408c23f68a5e2d5,POS,MCDONALDS 200 KRAKOW 07,5814,FAST FOOD RESTAURANTS,30-415,Dzielnica IX Łagiewniki-Borek Fałęcki,17.545713,202501
6371465,66704092,954c28dfb456a467153ef207edd667c994e4462cf0929d8,POS,MPK KRAKOW 61603,4131,BUS LINES,30-405,Dzielnica IX Łagiewniki-Borek Fałęcki,2.208000,202501


In [110]:
dane_dzielnice = duckdb.sql(f""" SELECT 
    tran_id_raw, 
    A.pymt_crd_acct_num_raw, 
    transaction_type, 
    upper(mrch_nm_raw) as mrch_nm_raw, 
    mrch_catg_cd, mrch_catg_nm, 
    A.district as mrch_district,
    cs_tran_amt, 
    prch_mnth_id,
    B.district as card_district
    
    FROM dzielnice_sprzed A
    
    INNER JOIN dzielnice_kup B ON A.pymt_crd_acct_num_raw = B.pymt_crd_acct_num_raw """).to_df() 

In [111]:
dane_dzielnice

,tran_id_raw,pymt_crd_acct_num_raw,transaction_type,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_district,cs_tran_amt,prch_mnth_id,card_district
0,719139291,771773b54df2ac382913c940403690a6d0abcb9666303c6,POS,CARREFOUR PLP325,5411,GROCERY STORES/SUPERMARKETS,Dzielnica III Prądnik Czerwony,93.83264,202503,Dzielnica III Prądnik Czerwony
1,719218064,06748570d9370f3207e963c9e1280b2a404a56d3d5d1fd0,POS,AUCHAN POLSKA SP. Z O.O,5411,GROCERY STORES/SUPERMARKETS,Dzielnica I Stare Miasto,37.28208,202509,Dzielnica IX Łagiewniki-Borek Fałęcki
2,719218177,2993527861e862e7e2bd9410668037f50f238c88bb4c3b0,POS,SWIAT KORALI,5621,WOMENS READY TO WEAR STORES,Dzielnica XV Mistrzejowice,132.48000,202509,Dzielnica I Stare Miasto
3,800425030,21a1a3b98f3e27fc4aefa27b47988253c707706b1ccbe03,POS,ZABKA Z7123 K.1,5499,MISC FOOD STORES - DEFAULT,Dzielnica XIV Czyżyny,36.41176,202501,Dzielnica I Stare Miasto
4,800425095,bcd95d2ab571a5d50a3d6b774cada214118dca04b28ee50,POS,LIDL 2043,5411,GROCERY STORES/SUPERMARKETS,Dzielnica III Prądnik Czerwony,237.37104,202501,Dzielnica XV Mistrzejowice
...,...,...,...,...,...,...,...,...,...,...
4597114,608183669,f1ef68672d09b3ffebe4fb1a14ef42282792e8030e2808c,POS,ZABKA ZE882 K.2,5499,MISC FOOD STORES - DEFAULT,Dzielnica XV Mistrzejowice,98.46944,202604,Dzielnica I Stare Miasto
4597115,608183926,c686a501d0b751b609eb545e9a5e85c409dc07b62cc2789,POS,MARKET PUNKT,5499,MISC FOOD STORES - DEFAULT,Dzielnica III Prądnik Czerwony,42.19488,202604,Dzielnica IV Prądnik Biały
4597116,608184211,7f780fd198f760b8a2c6af7e017b6b31942601b9c8b42bf,POS,PL BK KRAKOW SLIWIAKA D,5814,FAST FOOD RESTAURANTS,Dzielnica XV Mistrzejowice,153.20760,202604,Dzielnica V Krowodrza
4597117,608184359,8d496eeef28025e2c1b4ebf54fa1d8acadba99962158a06,POS,SKLEP SPOZYWCZO-MONOPOL,5399,MISC GENERAL MERCHANDISE,Dzielnica III Prądnik Czerwony,13.76320,202604,Dzielnica IV Prądnik Biały


In [112]:
max_m = duckdb.sql(f"SELECT MAX(prch_mnth_id) FROM dane_dzielnice ")
max_m.fetchone()[0]

202606

In [113]:
min_m = duckdb.sql(f"SELECT ( strftime( strptime(MAX(prch_mnth_id)::VARCHAR, '%Y%m') - INTERVAL 11 MONTH, '%Y%m'))::INTEGER FROM dane_dzielnice ")
min_m.fetchone()[0]

202507

In [137]:
kombinacje = duckdb.sql(f""" SELECT 
    card_district, 
    mrch_district, 
    count(*) as liczba_transakcji, 
    sum(cs_tran_amt) as wartosc_transakcji
    
    FROM dane_dzielnice 
    
    WHERE prch_mnth_id BETWEEN 202507 AND 202606 and card_district <> mrch_district
    
    GROUP BY card_district, mrch_district
    ORDER BY liczba_transakcji DESC, wartosc_transakcji """).to_df()

In [138]:
kombinacje

,card_district,mrch_district,liczba_transakcji,wartosc_transakcji
0,Dzielnica XV Mistrzejowice,Dzielnica I Stare Miasto,229215,2.680913e+07
1,Dzielnica I Stare Miasto,Dzielnica XV Mistrzejowice,175923,2.079956e+07
2,Dzielnica III Prądnik Czerwony,Dzielnica I Stare Miasto,145507,1.503279e+07
3,Dzielnica III Prądnik Czerwony,Dzielnica XV Mistrzejowice,140749,1.589330e+07
4,Dzielnica XV Mistrzejowice,Dzielnica IV Prądnik Biały,95381,1.149902e+07
...,...,...,...,...
300,Dzielnica XIV Czyżyny,Dzielnica X Swoszowice,7,7.748645e+02
301,Dzielnica X Swoszowice,Dzielnica XVII Wzgórza Krzesławickie,6,1.645696e+02
302,Dzielnica X Swoszowice,Dzielnica VII Zwierzyniec,5,2.046411e+02
303,Dzielnica XVII Wzgórza Krzesławickie,Dzielnica VIII Dębniki,3,1.533456e+02


In [143]:
mrch_district = 'Dzielnica XV Mistrzejowice'
card_district = 'Dzielnica I Stare Miasto'

In [145]:
dane_kombinacja = duckdb.sql(f""" SELECT 
    tran_id_raw, 
    pymt_crd_acct_num_raw, 
    transaction_type, 
    mrch_nm_raw, 
    mrch_catg_cd, 
    mrch_catg_nm, 
    mrch_district,
    cs_tran_amt, 
    prch_mnth_id,
    card_district 
    
    FROM dane_dzielnice 
    WHERE mrch_district = '{mrch_district}' and card_district = '{card_district}'""").to_df()

In [146]:
dane_kombinacja

,tran_id_raw,pymt_crd_acct_num_raw,transaction_type,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_district,cs_tran_amt,prch_mnth_id,card_district
0,719218177,2993527861e862e7e2bd9410668037f50f238c88bb4c3b0,POS,SWIAT KORALI,5621,WOMENS READY TO WEAR STORES,Dzielnica XV Mistrzejowice,132.48000,202509,Dzielnica I Stare Miasto
1,801965690,07b986ced85286ac22f6c7a34077c6390f1630460b9f63e,POS,KASOWNIK ZTP KRAKOW K1607,4111,LOCAL COMMUTER TRANSPORT,Dzielnica XV Mistrzejowice,8.28000,202510,Dzielnica I Stare Miasto
2,801965774,0647f6bdb8f28371dcd40ff838e2971d4813f2a70140b69,POS,"HS""JUBILAT"" STOISKO NR",5499,MISC FOOD STORES - DEFAULT,Dzielnica XV Mistrzejowice,58.73280,202510,Dzielnica I Stare Miasto
3,807708458,52be41d07731517ddcdce599b0d2c299f4b6dddb82845e5,POS,JMIDF SP.Z.O.O.HEBE R554,5977,COSMETIC STORES,Dzielnica XV Mistrzejowice,460.66792,202506,Dzielnica I Stare Miasto
4,764309743,ecce100ff5c5a9bb83f457c6c79164f82214dc398b91212,POS,JMP S.A. BIEDRONKA 6450,5411,GROCERY STORES/SUPERMARKETS,Dzielnica XV Mistrzejowice,18.36320,202508,Dzielnica I Stare Miasto
...,...,...,...,...,...,...,...,...,...,...
238450,1408496067,63746a6224b3b56d58658031e64dc9d4568ec1344069a23,POS,STOKROTKA NR 1194,5411,GROCERY STORES/SUPERMARKETS,Dzielnica XV Mistrzejowice,23.79672,202603,Dzielnica I Stare Miasto
238451,1229616766,a8e8ceee995e03139f2189ed37effe04407fd219f41d84d,POS,MPK KRAKOW 060923,4111,LOCAL COMMUTER TRANSPORT,Dzielnica XV Mistrzejowice,19.87200,202605,Dzielnica I Stare Miasto
238452,176492208,bcae5322afeae33ffa957c25b559760c716b849b8c1beb7,POS,SBL-ICIMB,5950,GLASSWARE/CRYSTAL STORES,Dzielnica XV Mistrzejowice,59.80000,202604,Dzielnica I Stare Miasto
238453,176493165,0ddceb03f96aaae20a56a705489919d970880aef68ad93b,POS,ZABKA Z8827 K.2,5499,MISC FOOD STORES - DEFAULT,Dzielnica XV Mistrzejowice,15.15976,202604,Dzielnica I Stare Miasto


In [149]:
dane_kombinacja.to_csv("dane_kombinacja.csv", index=False)

In [147]:
pivot = pd.pivot_table(
    dane_kombinacja,
    values="cs_tran_amt",
    index="transaction_type",
    columns="mrch_catg_nm",
    aggfunc="sum"
)


In [148]:
pivot

mrch_catg_nm,ADVERTISING SERVICES,AGRICULTURAL CO-OPERATIVE,AMUSEMENT PARKS/CIRCUS,ANTIQUE SHOPS,ARCHITECTURAL/ENG/SURVEY,ART DEALERS & GALLERIES,AUTO BODY REPAIR SHOPS,AUTO SERVICE SHOPS/NON DEALER,AUTOMATED FUEL DISPENSERS,AUTOMOBILE RENTAL AGENCY,...,USED MERCHANDISE STORES,UTILITIES/ELEC/GAS/H2O/SANI,VARIETY STORES,VETERINARY SERVICES,VIDEO AMUSEMENT GAME SUPPLY,VIDEO GAME ARCADES/ESTABLISH,WATCH/CLOCK/JEWELRY REPAIR,WIRE TRANSFER MONEY ORDER,WOMENS ACCESS/SPECIALTY,WOMENS READY TO WEAR STORES
transaction_type,,,,,,,,,,,,,,,,,,,,,
ATM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,349.6000,NaN,NaN,NaN,NaN,NaN,3463.36944,NaN
POS,616.3816,454.48,34919.15568,925.49792,1219.644,885.04,36229.41048,84132.50408,124339.49224,11902.5,...,3000.58552,11135.23472,92774.7044,133223.19256,107.456,418.6,820.732,35652.64224,80942.52552,40842.84616
